<a href="https://colab.research.google.com/github/nehalahane22-sketch/Financial-Forecasting-Frontier-Distributed-ML/blob/main/Banking_data_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Name**    -





##### **Project Type**    - EDA/Regression/Classification/Unsupervised
##### **Contribution**    - Individual
##### **Done By** - Neha Lahane

In [72]:
!java -version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.11" 2026-04-21
OpenJDK Runtime Environment (build 21.0.11+10-1-22.04.2-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.11+10-1-22.04.2-Ubuntu, mixed mode, sharing)


In [73]:
!pip install pyspark

In [74]:
from pyspark.sql import SparkSession

In [75]:
spark = SparkSession.builder \
    .appName("BankProject") \
    .getOrCreate()

In [76]:
spark

In [77]:
df = spark.read.csv("/content/bank.csv", header=True, inferSchema=True)

df.show(5)

df.printSchema()

print("Rows:", df.count())

print("Columns:", len(df.columns))

print(df.columns)

+---+-----------+-------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+---+
|age|        job|marital|education|default|balance|housing|loan| contact|day|month|duration|campaign|pdays|previous|poutcome|  y|
+---+-----------+-------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+---+
| 30| unemployed|married|  primary|     no|   1787|     no|  no|cellular| 19|  oct|      79|       1|   -1|       0| unknown| no|
| 33|   services|married|secondary|     no|   4789|    yes| yes|cellular| 11|  may|     220|       1|  339|       4| failure| no|
| 35| management| single| tertiary|     no|   1350|    yes|  no|cellular| 16|  apr|     185|       1|  330|       1| failure| no|
| 30| management|married| tertiary|     no|   1476|    yes| yes| unknown|  3|  jun|     199|       4|   -1|       0| unknown| no|
| 59|blue-collar|married|secondary|     no|      0|    yes|  no| unknown|  5|  may|     22

In [78]:
# Perform descriptive statistics and analyze customer distribution by subscription, job, balance, and marital status.
from pyspark.sql.functions import *

df.describe().show()

df.groupBy("y").count().show()

df.groupBy("job").count().orderBy(desc("count")).show()

df.select(avg("balance")).show()

df.groupBy("marital").agg(avg("balance").alias("Average_Balance")).show()

+-------+------------------+-------+--------+---------+-------+------------------+-------+----+--------+------------------+-----+------------------+------------------+------------------+------------------+--------+----+
|summary|               age|    job| marital|education|default|           balance|housing|loan| contact|               day|month|          duration|          campaign|             pdays|          previous|poutcome|   y|
+-------+------------------+-------+--------+---------+-------+------------------+-------+----+--------+------------------+-----+------------------+------------------+------------------+------------------+--------+----+
|  count|              4521|   4521|    4521|     4521|   4521|              4521|   4521|4521|    4521|              4521| 4521|              4521|              4521|              4521|              4521|    4521|4521|
|   mean| 41.17009511170095|   NULL|    NULL|     NULL|   NULL|1422.6578190665782|   NULL|NULL|    NULL|15.9152842291528

From this EDA, we can conclude:

- The dataset contains 4,521 customer records.  
- The average customer is about 41 years old.  
- Most customers did not subscribe to the bank's term deposit, making the target variable highly imbalanced.  
- The most common occupations are Management, Blue-collar, and Technician.  
- The average account balance is approximately 1,422, with values ranging from -3,313 to 71,188, indicating significant variability and possible outliers.  
- Customer call duration and the number of campaign contacts also vary widely, suggesting they may be important predictive features.  

In [79]:
# Data quality check: identify missing (NULL) values and duplicate rows before ML modeling.
from pyspark.sql.functions import col, sum, when

df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

print("Original Rows :", df.count())
print("Unique Rows   :", df.dropDuplicates().count())

+---+---+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+---+
|age|job|marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|  y|
+---+---+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+---+
|  0|  0|      0|        0|      0|      0|      0|   0|      0|  0|    0|       0|       0|    0|       0|       0|  0|
+---+---+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+---+

Original Rows : 4521
Unique Rows   : 4521


Overall EDA Findings

From this code, we conclude:

✅ No missing values in any column.  
✅ No duplicate rows in the dataset.  
✅ Dataset has 4521 unique customer records.  
✅ The data is clean and ready for preprocessing and machine learning.  

In [80]:
# Encode categorical features and convert the target variable into a numeric label for Spark ML.
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)

In [81]:
categorical_cols = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "poutcome"
]

print(categorical_cols)

['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


In [82]:
label_indexer = StringIndexer(
    inputCol="y",
    outputCol="label"
)

df = label_indexer.fit(df).transform(df)

In [83]:
df.select("y", "label").show(10)

+---+-----+
|  y|label|
+---+-----+
| no|  0.0|
| no|  0.0|
| no|  0.0|
| no|  0.0|
| no|  0.0|
| no|  0.0|
| no|  0.0|
| no|  0.0|
| no|  0.0|
| no|  0.0|
+---+-----+
only showing top 10 rows


In [84]:
df.groupBy("y", "label").count().show()

+---+-----+-----+
|  y|label|count|
+---+-----+-----+
| no|  0.0| 4000|
|yes|  1.0|  521|
+---+-----+-----+



In [85]:
# Convert categorical columns into numeric indices using a Spark ML pipeline and verify the encoded features.
indexers = [
    StringIndexer(
        inputCol=column,
        outputCol=column + "_index",
        handleInvalid="keep"
    )
    for column in categorical_cols
]

In [86]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=indexers)

df = pipeline.fit(df).transform(df)

In [87]:
df.select(
    "job",
    "job_index",
    "marital",
    "marital_index"
).show(10, truncate=False)

+-------------+---------+-------+-------------+
|job          |job_index|marital|marital_index|
+-------------+---------+-------+-------------+
|unemployed   |8.0      |married|0.0          |
|services     |4.0      |married|0.0          |
|management   |0.0      |single |1.0          |
|management   |0.0      |married|0.0          |
|blue-collar  |1.0      |married|0.0          |
|management   |0.0      |single |1.0          |
|self-employed|6.0      |married|0.0          |
|technician   |2.0      |married|0.0          |
|entrepreneur |7.0      |married|0.0          |
|services     |4.0      |married|0.0          |
+-------------+---------+-------+-------------+
only showing top 10 rows


In [88]:
# Apply one-hot encoding to categorical features and verify their vector representation.
encoder = OneHotEncoder(
    inputCols=[c + "_index" for c in categorical_cols],
    outputCols=[c + "_vec" for c in categorical_cols]
)

In [89]:
from pyspark.ml import Pipeline

encoder_pipeline = Pipeline(stages=[encoder])

df = encoder_pipeline.fit(df).transform(df)

In [90]:
df.select(
    "job",
    "job_index",
    "job_vec"
).show(10, truncate=False)

+-------------+---------+--------------+
|job          |job_index|job_vec       |
+-------------+---------+--------------+
|unemployed   |8.0      |(12,[8],[1.0])|
|services     |4.0      |(12,[4],[1.0])|
|management   |0.0      |(12,[0],[1.0])|
|management   |0.0      |(12,[0],[1.0])|
|blue-collar  |1.0      |(12,[1],[1.0])|
|management   |0.0      |(12,[0],[1.0])|
|self-employed|6.0      |(12,[6],[1.0])|
|technician   |2.0      |(12,[2],[1.0])|
|entrepreneur |7.0      |(12,[7],[1.0])|
|services     |4.0      |(12,[4],[1.0])|
+-------------+---------+--------------+
only showing top 10 rows


In [91]:
numeric_cols = [
    "age",
    "balance",
    "day",
    "duration",
    "campaign",
    "pdays",
    "previous"
]

print(numeric_cols)

['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']


In [92]:
encoded_cols = [c + "_vec" for c in categorical_cols]

print(encoded_cols)

['job_vec', 'marital_vec', 'education_vec', 'default_vec', 'housing_vec', 'loan_vec', 'contact_vec', 'month_vec', 'poutcome_vec']


In [93]:
assembler = VectorAssembler(
    inputCols=numeric_cols + encoded_cols,
    outputCol="features"
)

df = assembler.transform(df)

In [94]:
# Assemble all encoded and numerical columns into a single feature vector
df.select("features", "label").show(5, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                                  |label|
+--------------------------------------------------------------------------------------------------------------------------+-----+
|(51,[0,1,2,3,4,5,15,19,24,26,29,30,32,43,47],[30.0,1787.0,19.0,79.0,1.0,-1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])        |0.0  |
|(51,[0,1,2,3,4,5,6,11,19,22,26,28,31,32,35,48],[33.0,4789.0,11.0,220.0,1.0,339.0,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])|0.0  |
|(51,[0,1,2,3,4,5,6,7,20,23,26,28,30,32,40,48],[35.0,1350.0,16.0,185.0,1.0,330.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]) |0.0  |
|(51,[0,1,2,3,4,5,7,19,23,26,28,31,33,38,47],[30.0,1476.0,3.0,199.0,4.0,-1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])         |0.0  |
|(51,[0,2,3,4,5,8,19,22,26,28,30,33,35,47],[59.0,5.0,226.0,1.0,-1.0,1.0,1.0,1.0,1.0

In [95]:
# Split the dataset into 80% training data and 20% testing data.
# The training data is used to train the machine learning model,
# while the testing data is used to evaluate its performance.

train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print("Training Rows:", train_df.count())
print("Testing Rows :", test_df.count())


Training Rows: 3662
Testing Rows : 859


In [96]:
# Logistic Regression is being used because this is a binary classification problem: the target column y contains yes / no.
# Goal: Bank Marketing dataset, where the goal is essentially to predict whether a customer will subscribe to a term deposit (y = yes or no).


In [97]:
# Import LogisticRegression from PySpark's machine learning classification module.
from pyspark.ml.classification import LogisticRegression


# Create a Logistic Regression classification model.
# 'features' contains the input variables used for prediction.
# 'label' contains the target variable that we want to predict.
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label"
)


# Train the Logistic Regression model using the training dataset.
# The model learns the relationship between the input features
# and the target label from the training data.
lr_model = lr.fit(train_df)


In [98]:
# Use the trained model to make predictions on the unseen test dataset.
predictions = lr_model.transform(test_df)


# Display the actual label, predicted label, and prediction probability.
# 'label'       = actual value
# 'prediction'  = value predicted by the model
# 'probability' = probability assigned to each class by the model.
predictions.select(
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)


+-----+----------+------------------------------------------+
|label|prediction|probability                               |
+-----+----------+------------------------------------------+
|0.0  |0.0       |[0.8870160782266916,0.1129839217733084]   |
|0.0  |0.0       |[0.6532804420202498,0.3467195579797502]   |
|0.0  |0.0       |[0.9913793772744686,0.008620622725531435] |
|0.0  |0.0       |[0.9876573400295836,0.012342659970416392] |
|0.0  |0.0       |[0.7255251633929541,0.27447483660704586]  |
|0.0  |0.0       |[0.964549520802096,0.035450479197904006]  |
|1.0  |0.0       |[0.9892586021354656,0.01074139786453443]  |
|0.0  |0.0       |[0.7076795489397993,0.29232045106020066]  |
|0.0  |0.0       |[0.9930091777752574,0.0069908222247425655]|
|0.0  |0.0       |[0.9733527623392888,0.026647237660711198] |
+-----+----------+------------------------------------------+
only showing top 10 rows


In [99]:
# Import PySpark evaluator for calculating classification metrics.
from pyspark.ml.evaluation import MulticlassClassificationEvaluator


# Calculate the overall accuracy of the Logistic Regression model.
# Accuracy represents the proportion of correctly classified observations.
accuracy = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(predictions)


# Calculate weighted precision.
# Precision measures how reliable the model's predictions are
# across the different classes.
precision = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
).evaluate(predictions)


# Calculate weighted recall.
# Recall measures how effectively the model identifies
# the actual observations belonging to each class.
recall = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
).evaluate(predictions)


# Calculate the F1 Score.
# F1 Score provides a balance between precision and recall.
f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
).evaluate(predictions)


# Display the evaluation results.
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)


# Count actual vs predicted values.
# This helps identify correct predictions and classification errors.
predictions.groupBy("label", "prediction").count().show()

Accuracy : 0.8952270081490105
Precision: 0.8760483369427381
Recall   : 0.8952270081490105
F1 Score : 0.8783016907544199
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  1.0|       1.0|   28|
|  0.0|       1.0|   19|
|  1.0|       0.0|   71|
|  0.0|       0.0|  741|
+-----+----------+-----+



Actual	Predicted	Count

*   0 → 0	True Negative (TN)	741
*   0 → 1	False Positive (FP)	19
*   1 → 0	False Negative (FN)	71
*   1 → 1	True Positive (TP)	28







Recall for the Positive Class:TP/(TP+FN)=28.3%

Precision for the Positive Class: TP/(TP+FP) = 59.6%

In [100]:
# Import Random Forest Classifier from PySpark.
# Random Forest is a supervised machine learning algorithm
# used for classification problems.
from pyspark.ml.classification import RandomForestClassifier


In [101]:
# Create the Random Forest classification model.
# labelCol = target variable we want to predict.
# featuresCol = input features used by the model.
# numTrees = number of decision trees in the Random Forest.
# seed = ensures reproducible results.
rf = RandomForestClassifier(

    labelCol="label",
    featuresCol="features",
    numTrees=20,
    seed=42
)


In [102]:
# Train the Random Forest model using the training dataset.
# The model learns patterns between the input features and target label.
rf_model = rf.fit(train_df)

# Use the trained model to predict outcomes for the unseen test dataset.
rf_predictions = rf_model.transform(test_df)


In [103]:
# Display the actual label, predicted label,
# and probability for the first 10 test records.
rf_predictions.select(
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)


+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|0.0  |0.0       |[0.8748716883713159,0.12512831162868412]|
|0.0  |0.0       |[0.8428546908286499,0.15714530917135014]|
|0.0  |0.0       |[0.9354241237817337,0.06457587621826634]|
|0.0  |0.0       |[0.9270923532888926,0.07290764671110744]|
|0.0  |0.0       |[0.8780261571115231,0.12197384288847697]|
|0.0  |0.0       |[0.9108118522492079,0.08918814775079202]|
|1.0  |0.0       |[0.9259814947210903,0.07401850527890971]|
|0.0  |0.0       |[0.8923021824043282,0.10769781759567183]|
|0.0  |0.0       |[0.9321982949157013,0.06780170508429864]|
|0.0  |0.0       |[0.8966851347806655,0.10331486521933456]|
+-----+----------+----------------------------------------+
only showing top 10 rows


In [104]:
# Evaluate the Random Forest model using the same metrics
# used for Logistic Regression.

accuracy_rf = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(rf_predictions)

precision_rf = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
).evaluate(rf_predictions)

recall_rf = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
).evaluate(rf_predictions)

f1_rf = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
).evaluate(rf_predictions)


# Display Random Forest evaluation results.
print("Random Forest Accuracy :", accuracy_rf)
print("Random Forest Precision:", precision_rf)
print("Random Forest Recall   :", recall_rf)
print("Random Forest F1 Score :", f1_rf)


Random Forest Accuracy : 0.8847497089639115
Random Forest Precision: 0.8436072606493217
Random Forest Recall   : 0.8847497089639116
Random Forest F1 Score : 0.8389964593065813


In [105]:
# Compare actual labels with model predictions and count each combination.
# label = actual class and prediction = model's predicted class.
# This output helps identify True Positives, True Negatives,
# False Positives, and False Negatives.

rf_predictions.groupBy(
    "label",
    "prediction"
).count().show()


+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  1.0|       1.0|    4|
|  0.0|       1.0|    4|
|  1.0|       0.0|   95|
|  0.0|       0.0|  756|
+-----+----------+-----+



| Actual              | Predicted |   Count |
| ------------------- | --------- | ------: |
| True Negative (TN)  | 0 → 0     | **756** |
| False Positive (FP) | 0 → 1     |   **4** |
| False Negative (FN) | 1 → 0     |  **95** |
| True Positive (TP)  | 1 → 1     |   **4** |


| Metric    | Logistic Regression | Random Forest | Better              |
| --------- | ------------------: | ------------: | ------------------- |
| Accuracy  |          **89.52%** |        88.47% | Logistic Regression |
| Precision |          **87.60%** |        84.36% | Logistic Regression |
| Recall    |          **89.52%** |        88.47% | Logistic Regression |
| F1 Score  |          **87.83%** |        83.90% | Logistic Regression |


Create a Streaming Folder

1. “Spark reads continuously arriving data”

Normally, we have a fixed dataset. We read it once, process it, and finish.

But in Structured Streaming, data can keep arriving over time.

For example, imagine a bank:

Customer makes a transaction → new data arrives
Another customer makes a transaction → more data arrives
More transactions happen → more data arrives

Spark can keep watching for new data and process it automatically.

In your exercise, the streaming folder represents the place where new transaction files arrive.

So the idea is:

New data arrives → Spark detects it → Spark processes it → results are available

This is called streaming data processing.

2. “Data can be divided into partitions”

Spark doesn't have to process all the data as one big piece.

It can divide the data into smaller pieces called partitions.

For example:

Without partitioning:

1000 records → 1 big piece

With partitioning:

1000 records → 4 smaller pieces

These pieces can potentially be processed at the same time, depending on the available computing resources.

This is called parallel processing.

In [134]:
# ============================================================
# 1. CREATE STREAMING INPUT FOLDER
# ============================================================

import os

# Create the folder that Spark will monitor for new CSV files.
# exist_ok=True prevents an error if the folder already exists.

os.makedirs("/content/stream_data", exist_ok=True)

print("Streaming folder created!")

Streaming folder created!


In [135]:
# ============================================================
# 2. SELECT REQUIRED COLUMNS AND WRITE THEM AS CSV
# ============================================================

# Select only the columns needed for our streaming example.
sample = df.select(
    "age",
    "balance",
    "duration",
    "campaign",
    "label"
)

# Save the selected data as CSV.
# header=True writes the column names into the CSV.
sample.write.mode("overwrite") \
    .option("header", True) \
    .csv("/content/sample_transactions")

In [136]:
# ============================================================
# 3. FIND THE GENERATED CSV FILE
# ============================================================

import glob

# Spark creates files with names such as:
# part-00000-xxxxx.csv
#
# Find the CSV file generated above.
csv_file = glob.glob(
    "/content/sample_transactions/*.csv"
)[0]

print("CSV file:", csv_file)


CSV file: /content/sample_transactions/part-00000-4579f936-8336-412b-bcf2-c93b3a0938e8-c000.csv


In [137]:
# ============================================================
# 4. GET THE DATAFRAME SCHEMA
# ============================================================

# Get column names and their data types.
# We will provide this schema to readStream.
schema = sample.schema

print(schema)


StructType([StructField('age', IntegerType(), True), StructField('balance', IntegerType(), True), StructField('duration', IntegerType(), True), StructField('campaign', IntegerType(), True), StructField('label', DoubleType(), False)])


In [138]:
# ============================================================
# 5. CREATE THE STREAMING DATAFRAME
# ============================================================

stream_df = spark.readStream \
    .schema(schema) \
    .option("header", True) \
    .csv("/content/stream_data")

# Spark will now monitor /content/stream_data
# for newly arriving CSV files.


In [140]:
# ============================================================
# 6. START THE STREAMING QUERY
# ============================================================

query = stream_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

In [141]:
# 7. COPY A CSV FILE INTO THE STREAMING FOLDER
# ============================================================

import shutil

# Copy the CSV file into the folder being monitored.
# This simulates a new file arriving in a real streaming system.
shutil.copy(
    csv_file,
    "/content/stream_data/transactions.csv"
)


'/content/stream_data/transactions.csv'

In [113]:
query.stop()

In [142]:
# Copy the CSV file into the folder being monitored.
# This simulates a new file arriving in a real streaming system.
import shutil
import os

if os.path.exists("/content/stream_data"):
    shutil.rmtree("/content/stream_data")

os.makedirs("/content/stream_data")

In [143]:
!ls /content/stream_data

In [144]:
# Check currently active streaming queries
[(q.name, q.id, q.isActive) for q in spark.streams.active]
for q in spark.streams.active:
    if q.name == "transactions":
        q.stop()

In [145]:
stream_df = spark.readStream \
    .schema(schema) \
    .option("header", True) \
    .csv("/content/stream_data")

query = stream_df.writeStream \
    .format("memory") \
    .queryName("transactions") \
    .outputMode("append") \
    .start()

This code waits for Spark to process the incoming file, displays the streaming data, and checks the streaming status and progress.
It then stops the stream and checks the number of partitions.
Finally, it changes the data from 1 partition to 4 partitions to demonstrate parallel processing.

In [149]:
# Wait 3 seconds so Spark gets enough time to detect and process the new CSV file.
import time
time.sleep(3)

In [147]:
import shutil

shutil.copy(
    csv_file,
    "/content/stream_data/transactions.csv"
)

'/content/stream_data/transactions.csv'

In [150]:
# Wait 5 seconds so Spark gets enough time to detect and process the new CSV file.
time.sleep(5)

In [126]:
# Display the processed streaming data stored in the "transactions" memory table.
spark.sql("SELECT * FROM transactions").show()

+---+-------+--------+--------+-----+
|age|balance|duration|campaign|label|
+---+-------+--------+--------+-----+
| 30|   1787|      79|       1|  0.0|
| 33|   4789|     220|       1|  0.0|
| 35|   1350|     185|       1|  0.0|
| 30|   1476|     199|       4|  0.0|
| 59|      0|     226|       1|  0.0|
| 35|    747|     141|       2|  0.0|
| 36|    307|     341|       1|  0.0|
| 39|    147|     151|       2|  0.0|
| 41|    221|      57|       2|  0.0|
| 43|    -88|     313|       1|  0.0|
| 39|   9374|     273|       1|  0.0|
| 43|    264|     113|       2|  0.0|
| 36|   1109|     328|       2|  0.0|
| 20|    502|     261|       1|  1.0|
| 31|    360|      89|       1|  0.0|
| 40|    194|     189|       2|  0.0|
| 56|   4073|     239|       5|  0.0|
| 37|   2317|     114|       1|  0.0|
| 25|   -221|     250|       1|  0.0|
| 31|    132|     148|       1|  0.0|
+---+-------+--------+--------+-----+
only showing top 20 rows


In [127]:
print(query.lastProgress)

{
    "id": "bddddbd7-b30c-431b-beac-2577184c1cc6",
    "runId": "16bccdd3-ee54-4019-b311-b568c8e1cf37",
    "name": "transactions",
    "timestamp": "2026-08-27T08:22:26.517Z",
    "batchId": 1,
    "batchDuration": 2,
    "numInputRows": 0,
    "inputRowsPerSecond": 0.0,
    "processedRowsPerSecond": 0.0,
    "durationMs": {
        "latestOffset": 2,
        "triggerExecution": 2
    },
    "stateOperators": [],
    "sources": [
        {
            "description": "FileStreamSource[file:/content/stream_data]",
            "startOffset": {
                "logOffset": 0
            },
            "endOffset": {
                "logOffset": 0
            },
            "latestOffset": null,
            "numInputRows": 0,
            "inputRowsPerSecond": 0.0,
            "processedRowsPerSecond": 0.0
        }
    ],
    "sink": {
        "description": "MemorySink",
        "numOutputRows": 0
    }
}


In [128]:
query.isActive

True

In [129]:
print("Default Partitions:", df.rdd.getNumPartitions())

Default Partitions: 1


In [130]:
df_parallel = df.repartition(4)

print("New Partitions:", df_parallel.rdd.getNumPartitions())

New Partitions: 4


In [131]:
df_parallel = df.repartition(4)

print("New Partitions:", df_parallel.rdd.getNumPartitions())

New Partitions: 4


📝 Conclusion of the Project  

> Conclusion:  
> In this project, machine learning models were developed to predict whether a bank customer would subscribe to a term deposit. The data was preprocessed and divided into training and testing datasets. Two classification algorithms, Logistic Regression and Random Forest, were trained and evaluated using Accuracy, Precision, Recall, F1 Score, and actual-versus-predicted classifications.

> Logistic Regression performed better than Random Forest, achieving 89.52% accuracy and an F1 score of 87.83%, compared with 88.47% accuracy and an F1 score of 83.90% for Random Forest. Therefore, Logistic Regression was selected as the better-performing model among the two.


> However, the confusion matrix revealed a significant class imbalance, with both models struggling to identify customers belonging to the positive class (1). This indicates that further improvements such as class weighting, resampling, threshold tuning, and hyperparameter optimization could improve the model's ability to identify potential subscribers.